In [2]:
import pandas as pd
from pathlib import Path

# Find project root
project_root = Path.cwd().parents[1]

# Dataset path
file_path = project_root / "data" / "NorthBridgeDataset.xlsx"

# Load Tickets sheet
tickets = pd.read_excel(file_path, sheet_name="Tickets")

print(tickets.shape)
tickets.head()

(3500, 16)


,TicketID,TicketReference,ClientID,PatientRef,CategoryID,PriorityID,Status,AssignedAgentID,CreatedAt,FirstResponseAt,ResolvedAt,SLADueAt,SLABreached,Channel,Description,ResolutionNotes
0,1,NB-2024-00001,70,PT-81496,5,3,Closed,111,2024-07-24 23:41,2024-07-25 02:27,2024-07-27 10:35,2024-07-27 23:41,0,Phone,Request for discharge summary for patient PT-8...,Insurance medical report completed and submitt...
1,2,NB-2024-00002,51,PT-99051,2,1,Closed,32,2024-11-06 23:45,2024-11-07 01:09,2024-11-07 04:44,2024-11-07 07:45,0,Portal,Query regarding invoice 28714 dated 09 Oct 202...,Charge dispute investigated. Original amount v...
2,3,NB-2024-00003,53,PT-24869,3,3,Closed,57,2024-08-01 05:39,2024-08-01 09:58,2024-08-02 14:35,2024-08-04 05:39,0,Portal,Pre-authorisation request for outpatient consu...,Pre-auth declined by insurer. Patient advised ...
3,4,NB-2024-00004,11,PT-55537,1,2,Closed,22,2024-12-03 13:22,2024-12-03 14:47,2024-12-04 07:32,2024-12-04 01:22,1,Form,Patient PT-55537 requesting addition of appoin...,Cancellation processed. New appointment booked...
4,5,NB-2023-00005,70,PT-50539,3,2,Resolved,52,2023-02-11 03:56,2023-02-11 06:27,2023-02-11 12:26,2023-02-12 03:56,0,Phone,Pre-authorisation request for surgical procedu...,Pre-authorisation confirmed with insurer. Refe...


In [3]:
date_columns = [
    "CreatedAt",
    "FirstResponseAt",
    "ResolvedAt",
    "SLADueAt"
]

for col in date_columns:
    tickets[col] = pd.to_datetime(tickets[col], errors="coerce")

In [4]:
tickets[date_columns].dtypes

CreatedAt          datetime64[us]
FirstResponseAt    datetime64[us]
ResolvedAt         datetime64[us]
SLADueAt           datetime64[us]
dtype: object

In [5]:
# Create a summary table showing the number and percentage
# of missing values in each column.
missing_summary = pd.DataFrame({

    # Count the number of missing values in each column.
    "Missing_Count": tickets.isnull().sum(),

    # Calculate the percentage of missing values in each column
    # and round the result to 2 decimal places.
    "Missing_Percentage": (tickets.isnull().mean() * 100).round(2)
})

# Display only columns that contain at least one missing value.
missing_summary[missing_summary["Missing_Count"] > 0]

,Missing_Count,Missing_Percentage
FirstResponseAt,46,1.31
ResolvedAt,199,5.69
ResolutionNotes,199,5.69


In [6]:
# Select tickets where ResolvedAt is missing.
missing_resolved = tickets[tickets["ResolvedAt"].isnull()]

# Count the ticket statuses for those records.
# This helps us determine whether the missing resolution dates
# are expected because the tickets are still unresolved.
missing_resolved["Status"].value_counts()

Status
Escalated         57
In Progress       52
Open              46
Pending Client    44
Name: count, dtype: int64

In [7]:
# Select tickets where ResolutionNotes is missing.
missing_resolution_notes = tickets[tickets["ResolutionNotes"].isnull()]

# Count the status of tickets with missing resolution notes.
# This checks whether the missing notes are associated
# with tickets that have not yet been resolved.
missing_resolution_notes["Status"].value_counts()

Status
Escalated         57
In Progress       52
Open              46
Pending Client    44
Name: count, dtype: int64

In [8]:
# Select tickets where FirstResponseAt is missing.
missing_first_response = tickets[tickets["FirstResponseAt"].isnull()]

# Count the status of tickets with no first response.
# This helps determine whether the missing timestamps
# are expected for tickets that have not yet received a response.
missing_first_response["Status"].value_counts()

Status
Open    46
Name: count, dtype: int64

In [9]:
# Count completely duplicated rows in the Tickets table.
# A result of 0 means there are no exact duplicate records.
duplicate_rows = tickets.duplicated().sum()

print("Duplicate rows:", duplicate_rows)


# Check whether TicketID contains duplicates.
# TicketID should uniquely identify every ticket.
duplicate_ticket_ids = tickets["TicketID"].duplicated().sum()

print("Duplicate TicketIDs:", duplicate_ticket_ids)

Duplicate rows: 0
Duplicate TicketIDs: 0


In [10]:
# Display all unique values in the Status column.
# This helps identify inconsistent spelling, capitalisation,
# or unwanted spaces in ticket status values.
print("Status values:")
print(tickets["Status"].value_counts(dropna=False))


# Display all unique values in the Channel column.
# This helps identify inconsistent values such as
# "Email", "email", "EMAIL" or values containing extra spaces.
print("\nChannel values:")
print(tickets["Channel"].value_counts(dropna=False))

Status values:
Status
Closed            1984
Resolved          1317
Escalated           57
In Progress         52
Open                46
Pending Client      44
Name: count, dtype: int64

Channel values:
Channel
Email     1221
Portal     977
Phone      905
Form       397
Name: count, dtype: int64


In [11]:
# Check the unique values in PriorityID.
# This helps identify unexpected or invalid priority codes.
print("PriorityID values:")
print(sorted(tickets["PriorityID"].unique()))


# Check the unique values in CategoryID.
# This helps identify unexpected or invalid category codes.
print("\nCategoryID values:")
print(sorted(tickets["CategoryID"].unique()))


# Check the unique values in SLABreached.
# SLABreached is a binary target and should contain only 0 and 1.
print("\nSLABreached values:")
print(sorted(tickets["SLABreached"].unique()))

PriorityID values:
[np.int64(1), np.int64(2), np.int64(3), np.int64(4)]

CategoryID values:
[np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]

SLABreached values:
[np.int64(0), np.int64(1)]


In [12]:
# List the main text columns we want to inspect.
text_columns = [
    "TicketReference",
    "PatientRef",
    "Status",
    "Channel"
]

# Check each column for values containing leading
# or trailing whitespace.
for col in text_columns:
    whitespace_count = (
        tickets[col].astype(str) != tickets[col].astype(str).str.strip()
    ).sum()

    print(f"{col}: {whitespace_count} values with extra whitespace")

TicketReference: 0 values with extra whitespace
PatientRef: 0 values with extra whitespace
Status: 0 values with extra whitespace
Channel: 0 values with extra whitespace


In [13]:
# Check whether every TicketReference follows the expected format.
# Expected example: NB-2024-00001
invalid_ticket_refs = tickets[
    ~tickets["TicketReference"].str.match(r"^NB-\d{4}-\d{5}$", na=False)
]

print("Invalid TicketReference values:", len(invalid_ticket_refs))


# Check whether every PatientRef follows the expected format.
# Expected example: PT-81496
invalid_patient_refs = tickets[
    ~tickets["PatientRef"].str.match(r"^PT-\d{5}$", na=False)
]

print("Invalid PatientRef values:", len(invalid_patient_refs))

Invalid TicketReference values: 0
Invalid PatientRef values: 0


In [14]:
# List the key identifier columns that should always be populated.
key_columns = [
    "TicketID",
    "ClientID",
    "CategoryID",
    "PriorityID",
    "AssignedAgentID"
]

# Count missing values in each key column.
# These fields are important because they link tickets
# to clients, categories, priorities and agents.
for col in key_columns:
    missing_count = tickets[col].isnull().sum()
    print(f"{col}: {missing_count} missing values")

TicketID: 0 missing values
ClientID: 0 missing values
CategoryID: 0 missing values
PriorityID: 0 missing values
AssignedAgentID: 0 missing values


In [15]:
# Check whether any ticket received its first response
# before the ticket itself was created.
response_before_creation = tickets[
    tickets["FirstResponseAt"] < tickets["CreatedAt"]
]

# Check whether any ticket was resolved
# before the ticket itself was created.
resolution_before_creation = tickets[
    tickets["ResolvedAt"] < tickets["CreatedAt"]
]

# Check whether any SLA deadline occurs
# before the ticket was created.
sla_before_creation = tickets[
    tickets["SLADueAt"] < tickets["CreatedAt"]
]

# Display the number of problematic records found.
print("First response before creation:", len(response_before_creation))
print("Resolution before creation:", len(resolution_before_creation))
print("SLA due before creation:", len(sla_before_creation))

First response before creation: 0
Resolution before creation: 0
SLA due before creation: 0


In [16]:
# Select tickets that have a resolution timestamp.
# We only use resolved tickets for this check because
# unresolved tickets do not yet have a ResolvedAt value.
resolved_tickets = tickets[tickets["ResolvedAt"].notna()].copy()


# Calculate whether the resolution occurred after the SLA deadline.
# True means the ticket was resolved after SLADueAt.
resolved_tickets["CalculatedBreach"] = (
    resolved_tickets["ResolvedAt"] > resolved_tickets["SLADueAt"]
).astype(int)


# Compare the calculated result with the existing
# SLABreached field supplied in the dataset.
breach_mismatch = resolved_tickets[
    resolved_tickets["CalculatedBreach"] != resolved_tickets["SLABreached"]
]


# Display how many records do not agree.
print("Resolved tickets checked:", len(resolved_tickets))
print("SLABreached mismatches:", len(breach_mismatch))

Resolved tickets checked: 3301
SLABreached mismatches: 0


In [17]:
# Select tickets that have not yet been resolved.
# These records have no ResolvedAt timestamp.
unresolved_tickets = tickets[tickets["ResolvedAt"].isna()]


# Display the distribution of the SLA breach label
# among unresolved tickets.
# This helps us understand how SLABreached is being recorded
# for tickets that are still active.
print("Unresolved tickets:", len(unresolved_tickets))

print("\nSLABreached values for unresolved tickets:")
print(unresolved_tickets["SLABreached"].value_counts())

Unresolved tickets: 199

SLABreached values for unresolved tickets:
SLABreached
0    122
1     77
Name: count, dtype: int64


In [18]:
# Find the latest timestamp represented in the ticket dataset.
# This gives us a historical reference point for checking
# unresolved tickets rather than comparing them with today's date.
latest_dataset_date = tickets[
    ["CreatedAt", "FirstResponseAt", "ResolvedAt", "SLADueAt"]
].max().max()

print("Latest date in dataset:", latest_dataset_date)

Latest date in dataset: 2025-04-04 11:02:00


In [19]:
# Compare the SLA due date of unresolved tickets
# with the latest timestamp available in the dataset.
# This is only a diagnostic check.
# We are NOT changing the SLABreached target.

unresolved_tickets = unresolved_tickets.copy()

unresolved_tickets["DeadlinePassedByDatasetEnd"] = (
    unresolved_tickets["SLADueAt"] < latest_dataset_date
)

# Compare whether the SLA deadline had passed with
# the existing SLABreached value.
comparison = pd.crosstab(
    unresolved_tickets["DeadlinePassedByDatasetEnd"],
    unresolved_tickets["SLABreached"]
)

print(comparison)

SLABreached                   0   1
DeadlinePassedByDatasetEnd         
True                        122  77


In [20]:
# Confirm that temporary validation columns were not
# accidentally added to the main Tickets dataset.
print(tickets.columns.tolist())

['TicketID', 'TicketReference', 'ClientID', 'PatientRef', 'CategoryID', 'PriorityID', 'Status', 'AssignedAgentID', 'CreatedAt', 'FirstResponseAt', 'ResolvedAt', 'SLADueAt', 'SLABreached', 'Channel', 'Description', 'ResolutionNotes']


In [21]:
# Load the Clients and Agents sheets.
# These tables contain additional features that will later
# be joined to Tickets for SLA breach modelling.

clients = pd.read_excel(file_path, sheet_name="Clients")
agents = pd.read_excel(file_path, sheet_name="Agents")

# Confirm that both tables loaded successfully.
print("Clients shape:", clients.shape)
print("Agents shape:", agents.shape)

Clients shape: (80, 10)
Agents shape: (120, 8)


In [22]:
# Display the data types of all columns in the Clients table.
# This helps identify columns that need datatype conversion
# before the data is used for modelling.

clients.dtypes

ClientID             int64
ClientName             str
ClientType             str
ContractTier           str
AccountManagerID     int64
Region                 str
ContractStartDate      str
ContractEndDate        str
SLACreditClause      int64
IsActive             int64
dtype: object

In [23]:
# List the client date columns that need to be converted
# from strings into proper datetime values.
client_date_columns = [
    "ContractStartDate",
    "ContractEndDate"
]

# Convert each date column to datetime.
# errors="coerce" converts any invalid date value to NaT,
# allowing us to identify invalid dates afterwards.
for col in client_date_columns:
    clients[col] = pd.to_datetime(clients[col], errors="coerce")


# Check the data types after conversion.
print(clients[client_date_columns].dtypes)

ContractStartDate    datetime64[us]
ContractEndDate      datetime64[us]
dtype: object


In [24]:
# Check whether any client dates became missing after conversion.
# A value greater than 0 could indicate an invalid date
# in the original dataset.

print(
    "Invalid ContractStartDate:",
    clients["ContractStartDate"].isna().sum()
)

print(
    "Invalid ContractEndDate:",
    clients["ContractEndDate"].isna().sum()
)

Invalid ContractStartDate: 0
Invalid ContractEndDate: 0


In [25]:
# Check for clients whose contract end date occurs
# before their contract start date.
# Such records would represent an invalid contract period.

invalid_contract_dates = clients[
    clients["ContractEndDate"] < clients["ContractStartDate"]
]

# Display the number of invalid contract periods found.
print("Contracts ending before start date:", len(invalid_contract_dates))

Contracts ending before start date: 0


In [26]:
# Check the values used in SLACreditClause.
# This field should contain only 0 and 1.
print("SLACreditClause values:")
print(sorted(clients["SLACreditClause"].unique()))


# Check the values used in IsActive.
# This field should also contain only 0 and 1.
print("\nIsActive values:")
print(sorted(clients["IsActive"].unique()))

SLACreditClause values:
[np.int64(0), np.int64(1)]

IsActive values:
[np.int64(0), np.int64(1)]


In [27]:
# Display the unique values and counts for ContractTier.
# This helps identify spelling, capitalisation,
# or unexpected category values.
print("ContractTier values:")
print(clients["ContractTier"].value_counts(dropna=False))


# Display the unique values and counts for ClientType.
print("\nClientType values:")
print(clients["ClientType"].value_counts(dropna=False))


# Display the unique values and counts for Region.
print("\nRegion values:")
print(clients["Region"].value_counts(dropna=False))

ContractTier values:
ContractTier
Standard      37
Premium       29
Enterprise    14
Name: count, dtype: int64

ClientType values:
ClientType
GP Practice             25
Private Hospital        17
Specialist Clinic       15
Dental Group            10
Mental Health Clinic     8
Physiotherapy Centre     5
Name: count, dtype: int64

Region values:
Region
Midlands      21
Scotland      13
South East    12
North East    12
Yorkshire     11
North West    11
Name: count, dtype: int64


In [28]:
# Check whether ClientID contains duplicate values.
# ClientID should uniquely identify each client.
duplicate_client_ids = clients["ClientID"].duplicated().sum()

print("Duplicate ClientIDs:", duplicate_client_ids)


# List the main client text columns to inspect.
client_text_columns = [
    "ClientName",
    "ClientType",
    "ContractTier",
    "Region"
]

# Check for leading or trailing spaces in each text column.
for col in client_text_columns:
    whitespace_count = (
        clients[col].astype(str) != clients[col].astype(str).str.strip()
    ).sum()

    print(f"{col}: {whitespace_count} values with extra whitespace")

Duplicate ClientIDs: 0
ClientName: 0 values with extra whitespace
ClientType: 0 values with extra whitespace
ContractTier: 0 values with extra whitespace
Region: 0 values with extra whitespace


In [29]:
# Display the data types of all columns in the Agents table.
# This helps identify any columns that may need datatype conversion.

agents.dtypes

AgentID          int64
FullName           str
TeamID           int64
Role               str
Specialisms        str
Hub                str
DailyCapacity    int64
IsActive         int64
dtype: object

In [30]:
# Check the range of DailyCapacity.
# This helps identify impossible or suspicious values
# such as zero or negative agent capacity.
print("Minimum DailyCapacity:", agents["DailyCapacity"].min())
print("Maximum DailyCapacity:", agents["DailyCapacity"].max())


# Check the values used in IsActive.
# This should be a binary field containing only 0 and 1.
print("\nIsActive values:")
print(sorted(agents["IsActive"].unique()))


# Check whether AgentID contains duplicate values.
# AgentID should uniquely identify each agent.
print(
    "\nDuplicate AgentIDs:",
    agents["AgentID"].duplicated().sum()
)


# Check whether important agent fields contain missing values.
agent_key_columns = [
    "AgentID",
    "TeamID",
    "Role",
    "Hub",
    "DailyCapacity",
    "IsActive"
]

print("\nMissing values:")
print(agents[agent_key_columns].isnull().sum())

Minimum DailyCapacity: 6
Maximum DailyCapacity: 28

IsActive values:
[np.int64(0), np.int64(1)]

Duplicate AgentIDs: 0

Missing values:
AgentID          0
TeamID           0
Role             0
Hub              0
DailyCapacity    0
IsActive         0
dtype: int64


In [31]:
# Display the values and counts for agent Role.
# This helps identify inconsistent role names.
print("Role values:")
print(agents["Role"].value_counts(dropna=False))


# Display the values and counts for Hub.
# This helps identify inconsistent hub names.
print("\nHub values:")
print(agents["Hub"].value_counts(dropna=False))


# Check key text columns for leading or trailing spaces.
agent_text_columns = [
    "FullName",
    "Role",
    "Specialisms",
    "Hub"
]

for col in agent_text_columns:
    whitespace_count = (
        agents[col].astype(str) != agents[col].astype(str).str.strip()
    ).sum()

    print(f"{col}: {whitespace_count} values with extra whitespace")

Role values:
Role
Agent                 92
Senior Agent          18
Team Lead              9
Operations Manager     1
Name: count, dtype: int64

Hub values:
Hub
Manchester    42
Leeds         28
Birmingham    28
Edinburgh     22
Name: count, dtype: int64
FullName: 0 values with extra whitespace
Role: 0 values with extra whitespace
Specialisms: 0 values with extra whitespace
Hub: 0 values with extra whitespace


In [32]:
# Create a folder for cleaned data inside the local data directory.
# The data folder is ignored by Git, so local datasets will not be committed.
cleaned_data_dir = project_root / "data" / "cleaned"

cleaned_data_dir.mkdir(parents=True, exist_ok=True)


# Save each cleaned table as a CSV file.
# These files will be used in later EDA and modelling stages.

tickets.to_csv(
    cleaned_data_dir / "tickets_cleaned.csv",
    index=False
)

clients.to_csv(
    cleaned_data_dir / "clients_cleaned.csv",
    index=False
)

agents.to_csv(
    cleaned_data_dir / "agents_cleaned.csv",
    index=False
)

print("Cleaned datasets saved successfully.")

Cleaned datasets saved successfully.


In [33]:
# Confirm that each cleaned dataset was created successfully.

print(
    "Tickets:",
    (cleaned_data_dir / "tickets_cleaned.csv").exists()
)

print(
    "Clients:",
    (cleaned_data_dir / "clients_cleaned.csv").exists()
)

print(
    "Agents:",
    (cleaned_data_dir / "agents_cleaned.csv").exists()
)

Tickets: True
Clients: True
Agents: True


## Data Cleaning Summary

### Tickets
- Converted `CreatedAt`, `FirstResponseAt`, `ResolvedAt`, and `SLADueAt` from string to datetime.
- Investigated missing values in `FirstResponseAt`, `ResolvedAt`, and `ResolutionNotes`.
- Confirmed these missing values are valid business nulls associated with open or unresolved tickets.
- No duplicate rows were found.
- No duplicate `TicketID` values were found.
- `Status`, `Channel`, `PriorityID`, `CategoryID`, and `SLABreached` contain consistent and valid values.
- No unwanted leading or trailing whitespace was identified.
- `TicketReference` and `PatientRef` follow the expected formats.
- No invalid chronological relationships were identified.
- `SLABreached` was validated successfully for all 3,301 resolved tickets.
- Existing SLA breach labels for unresolved tickets were preserved.

### Clients
- Converted `ContractStartDate` and `ContractEndDate` from string to datetime.
- No invalid contract dates were found.
- No contract end dates occur before contract start dates.
- `SLACreditClause` and `IsActive` contain valid binary values.
- No duplicate `ClientID` values were found.
- Client categorical fields are consistent.
- No unwanted whitespace was identified.

### Agents
- Agent data types were already appropriate.
- `DailyCapacity` ranges from 6 to 28.
- `IsActive` contains valid binary values.
- No duplicate `AgentID` values were found.
- No important missing values were identified.
- Role and hub categories are consistent.
- No unwanted whitespace was identified.

### Cleaned Outputs
The cleaned datasets were saved locally as:

- `data/cleaned/tickets_cleaned.csv`
- `data/cleaned/clients_cleaned.csv`
- `data/cleaned/agents_cleaned.csv`

The local `data/` directory is excluded from Git so raw and cleaned datasets are not committed to the repository.

### Next Step
The cleaned datasets are ready for Exploratory Data Analysis (EDA).